<a href="https://colab.research.google.com/github/Melissa-Etes/ga4/blob/dev/ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ETL - GA4 Clinica

Le o export do GA4 (CSV), limpa/padroniza e devolve um dataframe pronto pra analise.

Este notebook detecta automaticamente se os dados ja vem limpos (ex: data em ISO `2026-07-01`,
duracao em segundos) ou "sujos" como um export real do GA4 costuma vir
(data em texto tipo `01 de jul de 2026`, duracao tipo `1m 06s`, nomes de coluna do relatorio,
linha de "Total", duplicatas). Assim o mesmo notebook funciona tanto com o CSV de exemplo
quanto com um export de verdade no futuro.

**Regra de ouro pra nao tomar `KeyError`/`NameError`: rode as celulas em ordem, de cima pra
baixo, sem pular nenhuma. Se algo der erro, va em Kernel > Restart e rode tudo de novo do
inicio (Run All) em vez de tentar consertar no meio.**

### 1. Imports

In [120]:
import pandas as pd
import numpy as np
import re

### 2. Extract — ler o CSV

In [121]:
# link do csv no GitHub
caminho_csv = "https://raw.githubusercontent.com/Melissa-Etes/ga4/dev/data/ga4_clinica_exemplo.csv"

df_bruto = pd.read_csv(caminho_csv, encoding="utf-8")
df_bruto.head()

,date,session_id,source,medium,campaign,device_category,sessions,engaged_sessions,page_views,session_duration_seconds,event_name,key_event_click_whatsapp
0,2026-07-01,100001,instagram,social,(not set),mobile,1,1,1,66,no_key_event,0
1,2026-07-01,100002,(direct),(none),(not set),mobile,1,1,2,58,no_key_event,0
2,2026-07-01,100003,google,cpc,Remarketing_Site,mobile,1,1,2,83,click_whatsapp,1
3,2026-07-01,100004,google,organic,(not set),mobile,1,0,1,25,no_key_event,0
4,2026-07-01,100005,google,cpc,Pesquisa_Implantes,tablet,1,0,1,3,no_key_event,0


### 3. Funcoes de limpeza

Cada funcao abaixo testa o formato do valor antes de converter — se o valor ja estiver
"limpo" (data ja em datetime/ISO, duracao ja numerica), ela so devolve o valor tratado sem
quebrar. Se vier "sujo" (texto tipo GA4 exporta), ela converte.

In [122]:
MESES = {
    "jan": 1, "fev": 2, "mar": 3, "abr": 4, "mai": 5, "jun": 6,
    "jul": 7, "ago": 8, "set": 9, "out": 10, "nov": 11, "dez": 12,
}

def parse_data(valor):
    """Aceita '01 de jul de 2026' (texto pt-br) OU '2026-07-01' (ISO) OU ja um Timestamp.
    Retorna sempre um pd.Timestamp (ou NaT se nao reconhecer o formato)."""
    if pd.isna(valor):
        return pd.NaT
    if isinstance(valor, pd.Timestamp):
        return valor
    valor = str(valor).strip()

    # formato texto pt-br: "01 de jul de 2026"
    m = re.match(r"(\d{1,2}) de (\w+) de (\d{4})", valor)
    if m:
        dia, mes_txt, ano = m.groups()
        mes = MESES.get(mes_txt.lower())
        if mes is not None:
            return pd.Timestamp(year=int(ano), month=mes, day=int(dia))

    # formato ISO ou qualquer outro que o pandas reconheca: "2026-07-01"
    return pd.to_datetime(valor, errors="coerce")


def parse_duracao(valor):
    """Aceita '1m 06s' (texto) OU um numero de segundos direto. Retorna sempre float (segundos)."""
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)

    valor = str(valor).strip()
    m = re.match(r"(\d+)m\s+(\d+)s", valor)
    if m:
        minutos, segundos = m.groups()
        return int(minutos) * 60 + int(segundos)

    # tenta converter direto pra numero (ex: coluna ja veio como "66")
    return pd.to_numeric(valor, errors="coerce")

Teste rapido das duas funcoes, pra confirmar que reconhecem os dois formatos antes de aplicar
na base inteira:

In [123]:
print(parse_data("01 de jul de 2026"))   # formato texto pt-br
print(parse_data("2026-07-01"))          # formato ISO (o do nosso CSV)
print(parse_duracao("1m 06s"))           # formato texto
print(parse_duracao(66))                 # formato numerico (o do nosso CSV)

2026-07-01 00:00:00
2026-07-01 00:00:00
66
66.0


### 4. Transform — funcao unica de limpeza


In [124]:
def transformar(df_bruto: pd.DataFrame) -> pd.DataFrame:
    df = df_bruto.copy()

    # 4.1 Padronizar nomes de coluna: minusculo, sem ponto/espaco, snake_case
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(".", "", regex=False)
        .str.replace(" ", "_", regex=False)
    )

    # 4.2 Renomear colunas do formato de relatorio do GA4 pro nosso padrao interno
    renomear = {
        "session_source": "source",
        "session_medium": "medium",
        "session_campaign": "campaign",
        "device_category": "device",
        "avg_session_duration": "duration_raw",
        "views": "page_views",
        "key_events": "key_event_click_whatsapp",
        "engaged_sessions": "engaged",
        "date": "date_raw",
        "session_duration_seconds": "duration_raw",
    }
    df = df.rename(columns={k: v for k, v in renomear.items() if k in df.columns})

    # 4.3 Remover linha de "Total" (comum em exports do GA4) e sessions sem id valido
    if "date_raw" in df.columns:
        df = df[df["date_raw"].astype(str).str.lower() != "total"]
    df["session_id"] = pd.to_numeric(df["session_id"], errors="coerce")
    df = df[df["session_id"].notna()]
    df["session_id"] = df["session_id"].astype(int)

    # 4.4 Data e duracao -> tipos corretos (funciona com texto pt-br OU ja limpo)
    df["date"] = df["date_raw"].apply(parse_data)
    df["duration_seconds"] = df["duration_raw"].apply(parse_duracao)
    df = df.drop(columns=[c for c in ["date_raw", "duration_raw"] if c in df.columns])

    # 4.5 Padronizar texto de origem (source)
    df["source"] = df["source"].astype(str).str.strip().str.lower()
    df["source"] = df["source"].replace({"google.com": "google"})

    # 4.6 Tratar valores ausentes
    df["campaign"] = df["campaign"].replace("", np.nan).fillna("(not set)")
    df["device"] = df["device"].replace("", np.nan).fillna("(not set)")

    # 4.7 Garantir tipos numericos corretos
    for col in ["sessions", "engaged", "page_views", "key_event_click_whatsapp"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

    # 4.8 Remover duplicatas (mesma sessao aparecendo mais de uma vez)
    df = df.drop_duplicates(subset=["session_id"])

    # 4.9 Reordenar colunas e ordenar por data
    ordem = [
        "date", "session_id", "source", "medium", "campaign", "device",
        "sessions", "engaged", "page_views", "duration_seconds",
        "event_name", "key_event_click_whatsapp",
    ]
    df = df[[c for c in ordem if c in df.columns]]
    df = df.sort_values("date").reset_index(drop=True)
    return df

### 5. Rodar a limpeza


In [125]:
df_limpo = transformar(df_bruto)

print(f"Linhas antes: {len(df_bruto)}")
print(f"Linhas depois: {len(df_limpo)}")
df_limpo.head()

Linhas antes: 2815
Linhas depois: 2815


,date,session_id,source,medium,campaign,device,sessions,engaged,page_views,duration_seconds,event_name,key_event_click_whatsapp
0,2026-07-01,100001,instagram,social,(not set),mobile,1,1,1,66.0,no_key_event,0
1,2026-07-01,100033,instagram,social,(not set),mobile,1,0,1,3.0,no_key_event,0
2,2026-07-01,100034,google,cpc,Remarketing_Site,mobile,1,0,2,35.0,no_key_event,0
3,2026-07-01,100035,instagram,social,(not set),mobile,1,0,2,96.0,no_key_event,0
4,2026-07-01,100036,(direct),(none),(not set),mobile,1,1,1,60.0,no_key_event,0


### 6. Load — salvar CSV limpo


In [126]:
df_limpo.to_csv("ga4_clinica_LIMPO.csv", index=False, encoding="utf-8-sig")

### 7. Analise rapida


In [127]:
total_sessoes = df_limpo["sessions"].sum()
total_conversoes = df_limpo["key_event_click_whatsapp"].sum()
print(f"Sessoes totais: {total_sessoes}")
print(f"Cliques no WhatsApp: {total_conversoes}")
print(f"Taxa de conversao geral: {100 * total_conversoes / total_sessoes:.2f}%")

resumo = (
    df_limpo.groupby(["source", "medium"])
    .agg(sessoes=("sessions", "sum"), conversoes=("key_event_click_whatsapp", "sum"))
    .assign(taxa_conversao=lambda d: (100 * d["conversoes"] / d["sessoes"]).round(2))
    .sort_values("sessoes", ascending=False)
)
resumo

Sessoes totais: 2815
Cliques no WhatsApp: 147
Taxa de conversao geral: 5.22%


sessoes  conversoes  taxa_conversao
source    medium                                      
google    cpc         1213          42            3.46
          organic      509          48            9.43
instagram social       381           5            1.31
facebook  cpc          271           1            0.37
(direct)  (none)       260          23            8.85
google    maps         181          28           15.47